In [25]:
import pandas as pd
import mlflow
import time

from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeRegressor
from mlflow.models import infer_signature

In [2]:
df = pd.read_csv("data/Position_Salaries.csv")

df.isna().any()

Position    False
Level       False
Salary      False
dtype: bool

In [3]:
# Data set already encodes position names to integers so drop non-numeric columns
numeric_df = df.select_dtypes(include="number")

X = numeric_df.iloc[:, :-1].to_numpy()
y = numeric_df.iloc[:, -1].to_numpy()

In [4]:
print(f"DataFrame shape: {df.shape}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

print("\nData types:")
for col, dtype in zip(df.columns, df.dtypes):
    print(col, dtype)

print("\n")
print(df.describe())

DataFrame shape: (10, 3)
X shape: (10, 1)
y shape: (10,)

Data types:
Position str
Level int64
Salary int64


          Level          Salary
count  10.00000       10.000000
mean    5.50000   249500.000000
std     3.02765   299373.883668
min     1.00000    45000.000000
25%     3.25000    65000.000000
50%     5.50000   130000.000000
75%     7.75000   275000.000000
max    10.00000  1000000.000000


In [28]:
class SklearnModelWrapper(mlflow.pyfunc.PythonModel):
  def __init__(self, model):
    self.model = model

  def predict(self, context, model_input, params=None):
    return self.model.predict(model_input)

with mlflow.start_run(run_name='untuned_decision_tree'):
    max_depth = 4
    regressor = DecisionTreeRegressor(random_state=0, max_depth=max_depth)
    regressor.fit(X, y)

    # Coefficient of determination R^2
    score = regressor.score(X, y)

    mlflow.log_param('max_depth', max_depth)
    mlflow.log_param("tree_depth", regressor.get_depth())
    mlflow.log_param("n_leaves", regressor.get_n_leaves())
    mlflow.log_metric('score', score)

    wrappedModel = SklearnModelWrapper(regressor)

    signature = infer_signature(X, wrappedModel.predict(None, X))

    mlflow.pyfunc.log_model(
        "decision_tree_model",
        python_model=wrappedModel,
        signature=signature
    )

/Users/dgshivu/Documents/lair/NEU/IE 7374/labs/lab2/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/10/18 19:13:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/18 19:13:05 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
/Users/dgshivu/Documents/lair/NEU/IE 7374/labs/lab2/.venv/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3344: Use

In [29]:
run_id = mlflow.search_runs(filter_string='tags.mlflow.runName = "untuned_decision_tree"').iloc[0].run_id
run_id

'60d2e8f48ac341c18207b988dae44480'

### Register Model

In [30]:
model_name = "position_salaries"
model_version = mlflow.register_model(f"runs:/{run_id}/decision_tree_model", model_name)

# Registering the model takes a few seconds, so add a small delay
time.sleep(15)

Registered model 'position_salaries' already exists. Creating a new version of this model...
2026/10/18 19:13:12 WARNING mlflow.tracking._model_registry.fluent: Run with id 60d2e8f48ac341c18207b988dae44480 has no artifacts at artifact path 'decision_tree_model', registering model based on models:/m-64d18028fc7a4a1fb96965ab825ff726 instead
Created version '6' of model 'position_salaries'.


### Check registered model

In [38]:
model = mlflow.pyfunc.load_model(f"models:/{model_name}/None")

# Sanity-check: This should match the R^2 logged by MLflow
print(f'Score: {r2_score(y, model.predict(X))}')

Score: 0.9998553639817056


In [ ]:
# !mlflow ui --port=5001

### Serve Registered Model

In [35]:
# !mlflow models serve --env-manager=local -m models:/position_salaries/None -h 0.0.0.0 -p 5001

#### Using the model served by MLFlow

In [36]:
import requests

url = 'http://localhost:5001/invocations'

req_body = req_body = {
    "dataframe_split": {
        "columns": ["Level"],
        "data": [[6.5], [4.0], [10]]
    }
}

response = requests.post(url, json=req_body)
predictions = response.json()

print(predictions)

{'predictions': [150000.0, 80000.0, 1000000.0]}
